In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from google.colab import files
files = files.upload()
NVDA = pd.read_csv("NVDA.csv")
NVDA.tail()


Saving NVDA_dataset.csv to NVDA_dataset (2).csv


,Date,Open,High,Low,Close,Volume
6531,2025-12-19,176.665,181.45,176.34,180.99,324925927
6532,2025-12-22,183.920,184.16,182.35,183.69,129064369
6533,2025-12-23,182.970,189.33,182.90,189.21,174873621
6534,2025-12-24,187.940,188.91,186.59,188.61,65528545
6535,2025-12-26,189.920,192.69,188.00,190.53,139740292


In [ ]:
X = NVDA.drop(columns=['Close','Date']).iloc[:-1].values
y = NVDA['Close'].iloc[1:].values
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=0)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform(X_test)


In [ ]:
from sklearn.linear_model import LinearRegression
Linear_model = LinearRegression()
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
poly = PolynomialFeatures(degree=2)
X_poly_train = poly.fit_transform(X_train_scaler)
X_test_poly = poly.transform(X_test_scaler)
Linear_model.fit(X_poly_train,y_train)
from sklearn import linear_model
Linear_model_lasso = linear_model.Lasso(alpha = 0.1, max_iter = 100, tol = 0.01)
Linear_model_lasso.fit(X_poly_train,y_train)
y_train_pred = Linear_model_lasso.predict(X_poly_train)
y_test_pred = Linear_model_lasso.predict(X_test_poly)
train_r2 = r2_score(y_train,y_train_pred)
test_r2 = r2_score(y_test,y_test_pred)
print("R2 for training data:", round(train_r2,4))
print("R2 for testing data:", round(test_r2,4))

R2 for training data: 0.9986
R2 for testing data: 0.9981


In [ ]:
NVDA['Date'] = pd.to_datetime(NVDA['Date'])
dates = NVDA['Date'].iloc[-len(y_test):]
graph_of_model = px.scatter(x=dates,y=y_test)
graph_of_model.add_scatter(x=dates,y=y_test_pred,mode='lines',name='Prediction Line')
graph_of_model.show()

In [ ]:
from sklearn.metrics import r2_score
model_accuracy = r2_score(y_test,y_test_pred)
model_accuracy=model_accuracy*100
print("The accuracy of this model is",round(model_accuracy,2),"%")

The accuracy of this model is 99.81 %


In [ ]:
X_next_day = NVDA[['Open','High','Low','Volume']].iloc[-1:].values
X_next_day_scaled = scaler.transform(X_next_day)
X_next_day_poly = poly.transform(X_next_day_scaled)
next_day_close = Linear_model_lasso.predict(X_next_day_poly)
print("The model's prediction for the next trading day(December 8 2025) is $",next_day_close.round(2))

The model's prediction for the next trading day(December 8 2025) is $ [189.92]


In [ ]:
# REMOVE REMINDER
tolerance = 3.00
correct = sum(abs(y_test - y_test_pred) <= tolerance)
total = len(y_test)
pseudo_accuracy = correct / total * 100

print(f"Pseudo-accuracy: {pseudo_accuracy:.2f}% within ${tolerance} tolerance")

Pseudo-accuracy: 96.48% within $3.0 tolerance
